# 01 · Fuentes y preparación

**Clases 2 y 3:** ETL y calidad de datos. Fuentes observadas: CAMMESA (energía mensual y potencia de cada parque solar argentino) y Open-Meteo (irradiación, temperatura y nubosidad del mes en cada ubicación).

Este notebook ejecuta el constructor real cuando faltan los procesados. Si ya están, los reutiliza.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    # Colab ya trae numpy, pandas, scikit-learn, xgboost y plotly: reinstalarlos rompe numpy en memoria.
    subprocess.run([sys.executable,'-m','pip','install','-q','py7zr','openpyxl'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import subprocess
esperado = DATA_PROCESSED / 'dataset_B_argentina_mensual.parquet'
if not esperado.exists():
    env = {**os.environ, 'TPO_DIR': str(DATA_PROCESSED), 'TPO_RAW': str(DATA_RAW)}
    subprocess.run([sys.executable, '-m', 'src.build_datasets'], cwd=ROOT, env=env, check=True)
else:
    print('Procesado disponible: se reutiliza la copia descargada.')
B = pd.read_parquet(esperado)
print(B.shape, f'{esperado.stat().st_size / 1e6:.2f} MB')

## Verificar la calidad antes de modelar

Un dato ausente no se reemplaza por cero. Un cero medido después del primer mes con generación es una parada real y se conserva. Las coordenadas deben venir de una fuente confiable y la potencia tiene que ser mayor que cero.

In [ ]:
print('Duplicados parque-mes:', B.duplicated(['central','fecha']).sum())
assert B.duplicated(['central','fecha']).sum() == 0
print('Faltantes por columna (las 10 mayores):')
print(B.isna().mean().sort_values(ascending=False).head(10).round(3))
print('Fuente de coordenadas:')
print(B.groupby('coord_source')['central'].nunique().sort_values(ascending=False))

In [ ]:
from src.academico import preparar, COORDS_OK, POTENCIA_DUDOSA
d = preparar(B)
pasos = pd.Series({
    'Original': len(B),
    'Coordenada confiable': int(B['coord_source'].isin(COORDS_OK).sum()),
    'Después de todas las reglas': len(d),
})
print(pasos)
print('Parques con potencia dudosa, solo entre 2022 y septiembre de 2025:', POTENCIA_DUDOSA)
print('Ceros operativos conservados:', int(d['energia_mwh'].eq(0).sum()))
print('Factor de capacidad: mínimo', d['factor_capacidad'].min(), '| máximo', round(d['factor_capacidad'].max(), 4))

## Fuentes

- CAMMESA: bases Energías Renovables 2026-09 y Generación Renovable Variable 2025-09.
- Weather data by Open-Meteo.com.
- `data/processed/diccionario_de_datos.md` documenta unidades y columnas.